In [ ]:
# ============================================================
# LOGISTIC REGRESSION
#
# ============================================================

!pip install -q gdown

import gdown

gdown.download(
    id="1DGDH2e6dEkpdwwtCvH6ds1CwJT2Hdmkv",
    output="HPV2025.xlsx",
    quiet=False
)

import os
import random
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    roc_auc_score,
    f1_score
)

from imblearn.over_sampling import SMOTE

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

def seed_everything(seed=43):

    random.seed(seed)
    np.random.seed(seed)

    os.environ["PYTHONHASHSEED"] = str(seed)

SEED = 43

seed_everything(SEED)

# ------------------------------------------------------------
# Read Dataset
# ------------------------------------------------------------

df = pd.read_excel("HPV2025.xlsx")

df = df.dropna(subset=["HPV Status"])

tobacco_mode = df["Tobacco Consumption"].mode()[0]

df["Tobacco Consumption"] = df[
    "Tobacco Consumption"
].fillna(tobacco_mode)

alcohol_mode = df["Alcohol Consumption"].mode()[0]

df["Alcohol Consumption"] = df[
    "Alcohol Consumption"
].fillna(alcohol_mode)

df = df.drop(
    columns=[
        "PatientID",
        "CenterID",
        "Task 1",
        "Task 2",
        "Task 3"
    ]
)

df = df.dropna()

print(df.shape)

# ------------------------------------------------------------
# Encode stages
# ------------------------------------------------------------

df["T-stage"] = df["T-stage"].replace({
    "T0":0,
    "T1":1,
    "T2":2,
    "T3":3,
    "T4":4
})

df["N-stage"] = df["N-stage"].replace({
    "N0":0,
    "N1":1,
    "N2":2,
    "N3":3
})

df["M-stage"] = df["M-stage"].replace({
    "M0":0,
    "M1":1
})

# ------------------------------------------------------------
# Features
# ------------------------------------------------------------

X = df[
[
"Age",
"Gender",
"Tobacco Consumption",
"Alcohol Consumption",
"Performance Status",
"Relapse",
"RFS",
"Treatment",
"T-stage",
"N-stage",
"M-stage"
]
]

y = df["HPV Status"]

# ------------------------------------------------------------
# Split
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.20,

    random_state=SEED

)

print("\nTrain samples:")

print(y_train.value_counts())

print("\nTest samples:")

print(y_test.value_counts())

# ------------------------------------------------------------
# Standardisation
# ------------------------------------------------------------

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)

# ------------------------------------------------------------
# SMOTE
# ------------------------------------------------------------

smote = SMOTE(random_state=SEED)

X_train_smote, y_train_smote = smote.fit_resample(

    X_train,
    y_train

)

print("\nAfter SMOTE:")

print(pd.Series(y_train_smote).value_counts())


# ------------------------------------------------------------
# Logistic Regression Model
# ------------------------------------------------------------

model = LogisticRegression(

    random_state=SEED,

    max_iter=1000

)

model.fit(

    X_train_smote,

    y_train_smote

)

# ------------------------------------------------------------
# Predictions
# ------------------------------------------------------------

predicted = model.predict(

    X_test

)

probabilities = model.predict_proba(

    X_test

)[:,1]

# ------------------------------------------------------------
# Classification Report
# ------------------------------------------------------------

print()

print("Classification Report\n")

print(

    classification_report(

        y_test,

        predicted,

        digits=4

    )

)

# ------------------------------------------------------------
# Confusion Matrix
# ------------------------------------------------------------

cm = confusion_matrix(

    y_test,

    predicted

)

print("Confusion Matrix")

print(cm)

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

accuracy = (

    (predicted == y_test).sum()

    /

    len(y_test)

)

bal_acc = balanced_accuracy_score(

    y_test,

    predicted

)

f1 = f1_score(

    y_test,

    predicted

)

auc = roc_auc_score(

    y_test,

    probabilities

)

print()

print(f"Accuracy:            {accuracy:.4f}")

print(f"Balanced Accuracy:   {bal_acc:.4f}")

print(f"F1-score:            {f1:.4f}")

print(f"AUC:                 {auc:.4f}")

# ------------------------------------------------------------
# MODEL SUMMARY
# ------------------------------------------------------------

print()

print("====================================")

print("LOGISTIC REGRESSION SUMMARY")

print("====================================")

print("Model : Logistic Regression")

print("Input Features : 11")

print("Learning Rate : N/A")

print("Optimiser : N/A")

print("Class Weights : None")

print("SMOTE : Yes")

print("Training Patients :", len(y_train))

print("Test Patients :", len(y_test))

print("Training after SMOTE :", len(y_train_smote))

print("Seed :", SEED)

print()

print("Final Results")

print("---------------------------")

print(f"Accuracy            : {accuracy:.4f}")

print(f"Balanced Accuracy   : {bal_acc:.4f}")

print(f"F1-score            : {f1:.4f}")

print(f"AUC                 : {auc:.4f}")

print()

print("Confusion Matrix")

print(cm)

print()

print("Analysis Complete.")

Downloading...
From: https://drive.google.com/uc?id=1DGDH2e6dEkpdwwtCvH6ds1CwJT2Hdmkv
To: /content/HPV2025.xlsx
100%|██████████| 66.0k/66.0k [00:00<00:00, 38.3MB/s]

(423, 12)

Train samples:
HPV Status
1.0    321
0.0     17
Name: count, dtype: int64

Test samples:
HPV Status
1.0    79
0.0     6
Name: count, dtype: int64

After SMOTE:
HPV Status
1.0    321
0.0    321
Name: count, dtype: int64

Classification Report

              precision    recall  f1-score   support

         0.0     0.3158    1.0000    0.4800         6
         1.0     1.0000    0.8354    0.9103        79

    accuracy                         0.8471        85
   macro avg     0.6579    0.9177    0.6952        85
weighted avg     0.9517    0.8471    0.8800        85

Confusion Matrix
[[ 6  0]
 [13 66]]

Accuracy:            0.8471
Balanced Accuracy:   0.9177
F1-score:            0.9103
AUC:                 0.8924

LOGISTIC REGRESSION SUMMARY
Model : Logistic Regression
Input Features : 11
Learning Rate : N/A
Optimiser : N/A
Class Weights : None
SMOTE : Yes
Training Patients : 338
Test Patients : 85
Training after SMOTE : 642
Seed : 43

Final Results
---------------------------
A


/tmp/ipykernel_592/687157250.py:88: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["T-stage"] = df["T-stage"].replace({
/tmp/ipykernel_592/687157250.py:96: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["N-stage"] = df["N-stage"].replace({
/tmp/ipykernel_592/687157250.py:103: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent